# Detector acceptance curves

This notebook plots the benchmark acceptance curves for DEAP-3600, DarkSide-50, DarkSide-20k, LZ, and XENONnT in independent figures.

The curves are approximate digitizations stored in `src.detectors.reference_detectors`. The DEAP curve is plotted in reconstructed electron-equivalent energy (`keVee`) using the 6.1 PE/keV mapping provided by the reference curve.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.detectors.reference_detectors import (
    _deap_acceptance,
    _ds20k_acceptance,
    _ds50_acceptance,
    _lz_acceptance,
    _xenont_acceptance,
)

## Prepare the curves

The curves are plotted using the energy variable provided by `reference_detectors`. For DEAP, this is reconstructed electron-equivalent energy (`keVee`) from the approximate PE mapping; no NR quenching-factor conversion is applied.

In [ ]:
acceptance_functions = {
    "DEAP-3600": _deap_acceptance,
    "DarkSide-50": _ds50_acceptance,
    "DarkSide-20k": _ds20k_acceptance,
    "LZ": _lz_acceptance,
    "XENONnT": _xenont_acceptance,
}

curves = {}
for detector, acceptance_function in acceptance_functions.items():
    acceptance = acceptance_function()
    energy_kev = acceptance.x / 1e3

    curves[detector] = (energy_kev, acceptance.y)

In [ ]:
x_limits = {
    "DEAP-3600": 200,
    "DarkSide-50": 1200,
    "DarkSide-20k": 1050,
    "LZ": 300,
    "XENONnT": 1050,
}

for detector, (energy_kev, acceptance) in curves.items():
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(
        energy_kev,
        acceptance,
        color="black",
        marker="o" if detector != "LZ" else None,
        label=detector,
    )

    ax.axvspan(30, 80, color="gray", alpha=0.08, label="30–80 keV reference band")
    ax.set(
        xlabel=(
            "Reconstructed energy [keVee] (rough PE mapping)"
            if detector == "DEAP-3600"
            else "Recoil energy [keV]"
        ),
        ylabel="Acceptance / efficiency",
        title=f"{detector} acceptance benchmark",
        xlim=(0, x_limits[detector]),
        ylim=(0, 1.02),
    )
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()